In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
# Chargement du fichier CSV
fichier_source = "log-selected-columns.csv"
df = pd.read_csv(fichier_source)

# Affichage des 5 premières lignes pour vérifier le chargement
print(f"Dimensions initiales du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

Dimensions initiales du dataset : 8739 lignes, 10 colonnes


,Depth,RxoRt,RLL3,SP,RILD,MN,MI,MCAL,DCAL,RHOB
0,195.0,226.0848,0.4076,69.8953,132.5052,-0.3548,0.1863,5.1090,1.8878,1.6973
1,195.5,223.5031,0.4063,69.2303,123.6225,-0.3619,0.1867,5.1031,1.8882,1.6791
2,196.0,221.4560,0.4047,68.4478,116.9258,-0.3668,0.1860,5.0872,1.8878,1.6585
3,196.5,219.8248,0.4033,67.4843,111.7925,-0.3613,0.1867,5.0881,1.8884,1.6435
4,197.0,218.1438,0.4023,66.3013,106.7821,-0.3569,0.1870,5.0972,1.8883,1.6390


In [2]:
# Création d'une copie pour ne pas altérer le DataFrame original
df_clean = df.copy()

# 1. Filtrage des résistivités (doivent être strictement positives)
# Les valeurs négatives indiquent des erreurs de mesure de l'outil.
df_clean = df_clean[(df_clean['RILD'] > 0) & 
                    (df_clean['RLL3'] > 0) & 
                    (df_clean['RxoRt'] > 0)]

# 2. Filtrage des densités (RHOB) réalistes
# On garde les valeurs entre 1.0 (eau) et 3.0 (roches très denses)
df_clean = df_clean[(df_clean['RHOB'] >= 1.0) & (df_clean['RHOB'] <= 3.0)]

# 3. Filtrage des effondrements de trou (Washouts)
# DCAL est la différence entre le diamètre réel et nominal. > 3 indique un effondrement.
df_clean = df_clean[df_clean['DCAL'] <= 3]

# Vérification du nombre de lignes conservées
print(f"Lignes conservées après nettoyage : {df_clean.shape[0]}")

Lignes conservées après nettoyage : 1241


In [3]:
# Calcul de la porosité à partir de la densité (RHOB)
# Hypothèses : Densité de la matrice (grès) = 2.65 g/cm³, Densité du fluide (eau) = 1.0 g/cm³
# Formule : Phi = (Densité_matrice - RHOB) / (Densité_matrice - Densité_fluide)
df_clean['Porosite'] = (2.65 - df_clean['RHOB']) / (2.65 - 1.0)

# Affichage des statistiques de la nouvelle colonne
df_clean['Porosite'].describe()

count    1241.000000
mean        0.501246
std         0.193112
min         0.139273
25%         0.354364
50%         0.452848
75%         0.633697
max         0.884545
Name: Porosite, dtype: float64

In [4]:
# La formule de porosité peut générer des valeurs aberrantes (ex: > 40% ou négatives)
# On filtre pour ne garder que des porosités réalistes pour un réservoir.
df_clean = df_clean[(df_clean['Porosite'] > 0) & (df_clean['Porosite'] <= 0.40)]

# Affichage du dataset final
print(f"Dimensions finales du dataset propre : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
df_clean.head()

Dimensions finales du dataset propre : 496 lignes, 11 colonnes


,Depth,RxoRt,RLL3,SP,RILD,MN,MI,MCAL,DCAL,RHOB,Porosite
49,224.5,13.9005,16.5665,-112.6851,22.4471,0.8505,0.4046,5.0853,1.8872,2.0445,0.366970
50,225.0,14.0429,15.8133,-114.4057,21.8853,0.8591,0.4007,5.0893,1.8886,2.1416,0.308121
51,225.5,15.4552,14.2983,-116.2029,21.7950,0.8621,0.4007,5.1064,1.8888,2.1834,0.282788
52,226.0,17.1174,12.8485,-118.0305,22.2077,0.8746,0.4074,5.0954,1.8889,2.1910,0.278182
53,226.5,17.5779,13.6895,-119.8371,22.7286,0.8918,0.4144,5.0872,1.8894,2.1862,0.281091


In [5]:
# Exportation du dataset propre vers un nouveau fichier CSV pour le prochain notebook
df_clean.to_csv("well_logs_clean.csv", index=False)
print("Fichier 'well_logs_clean.csv' exporté avec succès !")

Fichier 'well_logs_clean.csv' exporté avec succès !
